In [ ]:
import os

print(os.getcwd())
working_dir = '/content/drive/MyDrive/ML_Final'
os.chdir(working_dir)
print(os.getcwd())

/content
/content/drive/MyDrive/ML_Final


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
spark = SparkSession.builder \
    .appName("SpotifyRankPrediction") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

In [ ]:
spark = SparkSession.builder \
    .appName("Spotify Data Cleaned") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

# Read CSV with correct data types
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("samplingRatio", 1.0) \
    .option("nullValue", "null") \
    .option("treatEmptyValuesAsNulls", True) \
    .option("quote", '"') \
    .option("escape", '"') \
    .csv("2025_spotify_songs_cleaned.csv")

# Standardize dates
df = df.withColumn("snapshot_date", F.to_date("snapshot_date", "yyyy-MM-dd"))
df = df.withColumn("album_release_date", F.to_date("album_release_date", "yyyy-MM-dd"))

# Check schema
df.printSchema()

# Show the first 5 rows
df.show(5, truncate=False)


root
 |-- spotify_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- artists: string (nullable = true)
 |-- daily_rank: integer (nullable = true)
 |-- daily_movement: integer (nullable = true)
 |-- weekly_movement: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- snapshot_date: date (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- is_explicit: boolean (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- album_name: string (nullable = true)
 |-- album_release_date: date (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- key: integer (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: integer (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- tempo: double (nullable

## Using Viet Nam songs

In [ ]:
df = df[df['country'] == 'VN']

In [ ]:
print("Number of rows:", df.count())
print("Number of columns:", len(df.columns))
print("Columns information:", df.columns)


Number of rows: 7700
Number of columns: 25
Columns information: ['spotify_id', 'name', 'artists', 'daily_rank', 'daily_movement', 'weekly_movement', 'country', 'snapshot_date', 'popularity', 'is_explicit', 'duration_ms', 'album_name', 'album_release_date', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature']


In [ ]:
df = df.withColumn("snapshot_date", F.to_date("snapshot_date"))
df = df.withColumn("album_release_date", F.to_date("album_release_date"))

In [ ]:
w = Window.partitionBy("spotify_id").orderBy("snapshot_date")


## Label

In [ ]:
df = df.withColumn("next_day_rank", F.lead("daily_rank").over(w))
df = df.dropna(subset=["next_day_rank"])

## Feature Engineering

### Days since release

In [ ]:
df = df.withColumn(
    "days_since_release",
    F.datediff(F.col("snapshot_date"), F.col("album_release_date"))
)


### Rank fluctuation & popularity

In [ ]:
df = (
    df.withColumn("rank_diff", F.col("daily_rank") - F.lag("daily_rank", 1).over(w))
      .withColumn("popularity_diff", F.col("popularity") - F.lag("popularity", 1).over(w))
      .withColumn("rank_diff_7d", F.col("daily_rank") - F.lag("daily_rank", 7).over(w))
)


### Rolling 3-day Mean & Standard Deviation

In [ ]:
w3 = w.rowsBetween(-2, 0)

df = (
    df.withColumn("rank_rolling_mean_3d", F.avg("daily_rank").over(w3))
      .withColumn("popularity_rolling_mean_3d", F.avg("popularity").over(w3))
      .withColumn("rank_rolling_std_3d", F.stddev("daily_rank").over(w3))
      .withColumn("popularity_rolling_std_3d", F.stddev("popularity").over(w3))
)


### Split Day & Month (Period)

In [ ]:
df = (
    df.withColumn("day_of_week", F.dayofweek("snapshot_date"))  # 1=Sunday,...7=Saturday
      .withColumn("month", F.month("snapshot_date"))
)


### All features

In [ ]:
feature_cols = [
    'daily_rank', 'daily_movement', 'weekly_movement',
    'popularity', 'popularity_diff', 'popularity_rolling_mean_3d',
    'rank_diff', 'rank_rolling_mean_3d',
    'rank_diff_7d', 'rank_rolling_std_3d', 'popularity_rolling_std_3d',
    'danceability', 'energy', 'speechiness', 'acousticness',
    'instrumentalness', 'liveness', 'valence', 'tempo',
    'days_since_release', 'day_of_week', 'month'
]


### Remove null values since lag operation


In [ ]:
df = df.na.fill(0, subset=feature_cols)

### Check with example data

In [ ]:
df.select("spotify_id", "snapshot_date", "daily_rank",
          "rank_diff", "rank_rolling_mean_3d", "popularity_diff",
          "days_since_release", "day_of_week", "month").show(10)


+--------------------+-------------+----------+---------+--------------------+---------------+------------------+-----------+-----+
|          spotify_id|snapshot_date|daily_rank|rank_diff|rank_rolling_mean_3d|popularity_diff|days_since_release|day_of_week|month|
+--------------------+-------------+----------+---------+--------------------+---------------+------------------+-----------+-----+
|015cWxefajYwEUZod...|   2025-02-09|        37|        0|                37.0|              0|                10|          1|    2|
|015cWxefajYwEUZod...|   2025-02-10|        49|       12|                43.0|              1|                11|          2|    2|
|015cWxefajYwEUZod...|   2025-02-11|        46|       -3|                44.0|              1|                12|          3|    2|
|015cWxefajYwEUZod...|   2025-02-12|        36|      -10|  43.666666666666664|              1|                13|          4|    2|
|015cWxefajYwEUZod...|   2025-02-13|        44|        8|                42.

## TRAIN

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler

In [ ]:
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="unscaled_features"
)

scaler = StandardScaler(
    inputCol="unscaled_features",
    outputCol="features"
)


In [ ]:
df_sorted = df.orderBy("snapshot_date")

total_rows = df_sorted.count()

# 80% Position
split_point = int(total_rows * 0.8)
print(f"Tổng dòng: {total_rows}, Train: {split_point}, Test: {total_rows - split_point}")


Tổng dòng: 7531, Train: 6024, Test: 1507


In [ ]:
# 80% First for training
train_df = df_sorted.limit(split_point)

# 20% End for testing
test_df = df_sorted.subtract(train_df)

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())


Train rows: 6024
Test rows: 1507


In [ ]:
print("Train date range:")
train_df.agg(F.min("snapshot_date"), F.max("snapshot_date")).show()

print("Test date range:")
test_df.agg(F.min("snapshot_date"), F.max("snapshot_date")).show()


Train date range:
+------------------+------------------+
|min(snapshot_date)|max(snapshot_date)|
+------------------+------------------+
|        2025-01-01|        2025-05-11|
+------------------+------------------+

Test date range:
+------------------+------------------+
|min(snapshot_date)|max(snapshot_date)|
+------------------+------------------+
|        2025-05-11|        2025-06-10|
+------------------+------------------+



In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from xgboost.spark import SparkXGBRegressor

In [ ]:
# 1. Linear Regression
lr = LinearRegression(featuresCol="features", labelCol="next_day_rank")
lr_grid = (ParamGridBuilder()
           .addGrid(lr.regParam, [0.1, 0.01])  # Try 2 regularization values
           .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) # Try 3 types (L2, Elastic, L1)
           .build())

# 2. Random Forest
rf = RandomForestRegressor(featuresCol="features", labelCol="next_day_rank", seed=42)
rf_grid = (ParamGridBuilder()
           .addGrid(rf.numTrees, [50, 100]) # Try 2 numbers of trees
           .addGrid(rf.maxDepth, [5, 10])     # Try 2 depths
           .build())

# 3. XGBoost
xgb = SparkXGBRegressor(objective="reg:squarederror", features_col="features", label_col="next_day_rank", seed=42)
xgb_grid = (ParamGridBuilder()
            .addGrid(xgb.n_estimators, [100, 200]) # Try 2 numbers of estimators (trees)
            .addGrid(xgb.max_depth, [4, 6])        # Try 2 depths
            .addGrid(xgb.learning_rate, [0.1, 0.05]) # Try 2 learning rates
            .build())

In [ ]:
LABEL_COLUMN_NAME = "next_day_rank"

evaluator_rmse = RegressionEvaluator(labelCol=LABEL_COLUMN_NAME,
                                       predictionCol="prediction",
                                       metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol=LABEL_COLUMN_NAME,
                                     predictionCol="prediction",
                                     metricName="r2")

In [ ]:
results = {}

In [ ]:
models_to_tune = [
    ("Linear Regression", lr, lr_grid),
    ("Random Forest", rf, rf_grid),
    ("XGBoost", xgb, xgb_grid)
]

In [ ]:
for name, model, grid in models_to_tune:
    print(f"--- Starting to 'buff' model: {name} ---")

    # 1. Create the complete Pipeline
    # NOTE: 'assembler' and 'scaler' must be defined before this part
    pipeline = Pipeline(stages=[assembler, scaler, model])

    # 2. Create CrossValidator
    # It will automatically split train_df into 3 parts (numFolds=3)
    # and find the best parameter set (estimatorParamMaps=grid)
    cv = CrossValidator(estimator=pipeline,
                        estimatorParamMaps=grid,
                        evaluator=evaluator_rmse,
                        numFolds=3) # Use 3-fold CV. Increase to 5 or 10 for more thorough tuning

    # 3. Train the CrossValidator
    # This is where the time-consuming "tuning" process happens
    cv_model = cv.fit(train_df)

    # 4. Get the best model
    best_model = cv_model.bestModel

    # 5. Evaluate on the Test set (data the model has never seen)
    predictions = best_model.transform(test_df)
    rmse = evaluator_rmse.evaluate(predictions)
    r2 = evaluator_r2.evaluate(predictions)

    # 6. Save and print the result
    results[name] = {'RMSE':rmse, 'R2': r2}
    print(f"Successfully 'buffed' {name}. RMSE on Test set: {rmse}\n")

    # (Optional) Print the best parameters it found
    # Get the model stage (the last stage of the pipeline)
    best_model_stage = best_model.stages[-1]
    print(f"Best parameters for {name}:")
    # Get the parameters that were set in the grid
    best_params = {param.name: best_model_stage.getOrDefault(param) for param in grid[0]}
    print(best_params, "\n" + "="*30)

--- Starting to 'buff' model: Linear Regression ---
Successfully 'buffed' Linear Regression. RMSE on Test set: 5.98851340492181

Best parameters for Linear Regression:
{'regParam': 0.1, 'elasticNetParam': 1.0} 
--- Starting to 'buff' model: Random Forest ---
Successfully 'buffed' Random Forest. RMSE on Test set: 5.967801491776169

Best parameters for Random Forest:
{'numTrees': 100, 'maxDepth': 10} 
--- Starting to 'buff' model: XGBoost ---


INFO:XGBoost-PySpark:Running xgboost-3.0.5 on 1 workers with
	booster params: {'device': 'cpu', 'learning_rate': 0.1, 'max_depth': 4, 'objective': 'reg:squarederror', 'seed': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!
INFO:XGBoost-PySpark:Running xgboost-3.0.5 on 1 workers with
	booster params: {'device': 'cpu', 'learning_rate': 0.05, 'max_depth': 4, 'objective': 'reg:squarederror', 'seed': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!
INFO:XGBoost-PySpark:Running xgboost-3.0.5 on 1 workers with
	booster params: {'device': 'cpu', 'learning_rate': 0.1, 'max_depth': 6, 'objective': 'reg:squarederror', 'seed': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatr

Successfully 'buffed' XGBoost. RMSE on Test set: 5.930856635949452

Best parameters for XGBoost:
{'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.1} 


In [ ]:
print("\n--- FINAL 'BUFF' RESULTS ---")
for model_name, metrics in results.items():
    print(f"{model_name}: RMSE = {metrics['RMSE']}, R2 = {metrics['R2']}")


--- FINAL 'BUFF' RESULTS ---
Linear Regression: RMSE = 5.98851340492181, R2 = 0.8249854105774537
Random Forest: RMSE = 5.967801491776169, R2 = 0.8261939303556698
XGBoost: RMSE = 5.930856635949452, R2 = 0.8283392310044118


## Notebook Summary

This notebook focuses on building and evaluating machine learning models to predict the next day's rank of Spotify songs, specifically for songs from Vietnam.

**Key Steps:**

1.  **Environment Setup:**
    *   Changed the working directory to `/content/drive/MyDrive/ML_Final`.
    *   Initialized a SparkSession for distributed computing.

2.  **Data Loading and Preparation:**
    *   Loaded the `2025_spotify_songs_cleaned.csv` file into a Spark DataFrame.
    *   Standardized date formats for `snapshot_date` and `album_release_date`.
    *   Filtered the dataset to include only songs from Vietnam ('VN').
    *   Checked the dimensions and schema of the filtered DataFrame.

3.  **Feature Engineering:**
    *   Created a window specification partitioned by `spotify_id` and ordered by `snapshot_date` for time-series operations.
    *   Labeled the data by adding a `next_day_rank` column using the `lead` function. Rows with no next day rank were dropped.
    *   Engineered several features:
        *   `days_since_release`: Number of days between the snapshot date and album release date.
        *   `rank_diff` and `popularity_diff`: Daily changes in rank and popularity.
        *   `rank_diff_7d`: Rank change over a 7-day period.
        *   `rank_rolling_mean_3d`, `popularity_rolling_mean_3d`: Rolling 3-day mean of rank and popularity.
        *   `rank_rolling_std_3d`, `popularity_rolling_std_3d`: Rolling 3-day standard deviation of rank and popularity.
        *   `day_of_week` and `month`: Extracted day of the week and month from the snapshot date.
    *   Defined a list of `feature_cols` to be used for modeling.
    *   Filled any remaining null values in the feature columns with 0.

4.  **Data Splitting:**
    *   Sorted the data by `snapshot_date`.
    *   Split the data into training (80%) and testing (20%) sets based on time, ensuring that the test set contains data from later dates than the training set.

5.  **Model Training and Evaluation:**
    *   Defined `VectorAssembler` and `StandardScaler` for feature processing within a pipeline.
    *   Set up three regression models for tuning:
        *   Linear Regression (`LinearRegression`)
        *   Random Forest Regressor (`RandomForestRegressor`)
        *   XGBoost Regressor (`SparkXGBRegressor`)
    *   Defined parameter grids for tuning each model.
    *   Initialized `RegressionEvaluator` for RMSE and R2 metrics.
    *   Used `CrossValidator` with 3-fold cross-validation to find the best parameters for each model on the training data.
    *   Evaluated the best model for each algorithm on the unseen test data using RMSE and R2.
    *   Stored the evaluation results in a dictionary.

6.  **Results:**
    *   Printed the final RMSE and R2 scores for each trained model on the test set.

This notebook successfully preprocesses the Spotify data, engineers relevant features, and trains and evaluates three different regression models for predicting song rank.